# Тема 2. Визуальный анализ данных

**Цель:** научиться быстро понимать данные с помощью графиков — ещё до того, как строить модели.

Визуализация нужна на всех этапах ML-проекта:
- при первичном знакомстве с данными
- при отборе признаков
- при интерпретации результатов модели

Будем использовать три библиотеки: `pandas` (встроенные графики), `matplotlib` (основа), `seaborn` (красивые статистические графики поверх matplotlib).

### Содержание
1. [Датасет](#1.-Датасет)
2. [Одномерный анализ](#2.-Одномерный-анализ)  
3. [Многомерный анализ](#3.-Многомерный-анализ)
4. [Визуализация всего датасета](#4.-Визуализация-всего-датасета)
5. [Самостоятельная работа: Titanic](#5.-Самостоятельная-работа:-Titanic)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()  # делает все графики matplotlib сразу красивее

# SVG — чёткие и масштабируемые графики
%config InlineBackend.figure_format = 'svg'

## 1. Датасет

Будем работать с данными телеком-оператора: 3333 клиента, признаки об их активности и целевая переменная `Churn` — ушёл ли клиент.

In [ ]:
df = pd.read_csv("../../data/telecom_churn.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

Целевая переменная — **`Churn`** (булева): `True` — клиент ушёл, `False` — остался.

Признаки делятся на:
- **количественные** (`Total day minutes`, `Account length`, ...) — числа
- **категориальные** (`State`, `International plan`, ...) — группы

---
## 2. Одномерный анализ

Смотрим на каждый признак по отдельности: какое распределение, есть ли выбросы, насколько сбалансированы классы.

### 2.1 Количественные признаки

#### Гистограмма

Самый простой способ посмотреть на распределение числовой переменной.

In [ ]:
features = ["Total day minutes", "Total intl calls"]
df[features].hist(figsize=(10, 4));

*Total day minutes* — почти нормальное распределение.  
*Total intl calls* — скошено вправо (длинный хвост справа).

#### Оценка плотности (KDE)

Сглаженная версия гистограммы — не зависит от ширины бинов.

In [ ]:
df[features].plot(kind="density", subplots=True, layout=(1, 2), sharex=False, figsize=(10, 4));

Или через seaborn — сразу гистограмма + KDE:

In [ ]:
sns.histplot(df["Total intl calls"], kde=True);

#### Box plot (ящик с усами)

Показывает медиану, квартили и выбросы одним взглядом.

In [ ]:
sns.boxplot(x="Total intl calls", data=df);

Как читать box plot:

```
       |----[   |   ]---|  o  o
      min  Q1  med Q3  max  outliers
```

- **Ящик**: от Q1 (25%) до Q3 (75%) — интерквартильный размах IQR  
- **Линия внутри**: медиана (50%)  
- **Усы**: Q1 − 1.5·IQR и Q3 + 1.5·IQR  
- **Точки**: выбросы — значения вне усов

> 💬 **Покажи сам:** построй box plot для `Total day minutes`

In [ ]:
# место для демонстрации

### 2.2 Категориальные признаки

#### Таблица частот

In [ ]:
df["Churn"].value_counts()

Классы **несбалансированы**: лояльных клиентов ~86%, ушедших ~14%. Это важно при построении моделей.

#### Bar plot (столбчатая диаграмма)

In [ ]:
_, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="Churn", data=df, ax=axes[0])
sns.countplot(x="Customer service calls", data=df, ax=axes[1]);

Отличие от гистограммы: bar plot — для категорий, гистограмма — для чисел.

> 💬 **Покажи сам:** посмотри на распределение `International plan` через `countplot`

In [ ]:
# место для демонстрации

---
## 3. Многомерный анализ

Ищем связи между признаками и с целевой переменной.

### 3.1 Количественный vs. Количественный

#### Корреляционная матрица

In [ ]:
# Оставим только числовые столбцы
numerical = df.select_dtypes(include="number").columns.tolist()
numerical.remove("Area code")  # это категория, несмотря на тип int

corr_matrix = df[numerical].corr()
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm");

Видно, что `Total day charge` посчитан прямо из `Total day minutes` — такие зависимые признаки избыточны.

In [ ]:
# Уберём производные признаки
redundant = ["Total day charge", "Total eve charge", "Total night charge", "Total intl charge"]
numerical_clean = [c for c in numerical if c not in redundant]

sns.heatmap(df[numerical_clean].corr(), annot=True, fmt=".2f", cmap="coolwarm", figsize=(10, 8));

#### Scatter plot

In [ ]:
plt.scatter(df["Total day minutes"], df["Total night minutes"], alpha=0.3);

In [ ]:
# Или через seaborn с гистограммами по краям
sns.jointplot(x="Total day minutes", y="Total night minutes", data=df, kind="scatter");

### 3.2 Количественный vs. Категориальный

Это самое важное в разведочном анализе для классификации: как числовые признаки ведут себя в разных классах?

#### Box plots по целевой переменной

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))

for idx, feat in enumerate(numerical_clean):
    ax = axes[idx // 4, idx % 4]
    sns.boxplot(x="Churn", y=feat, data=df, ax=ax)
    ax.set_xlabel("")
    ax.set_ylabel(feat, fontsize=9)

# убираем лишние оси
for idx in range(len(numerical_clean), 12):
    axes[idx // 4, idx % 4].set_visible(False)

fig.tight_layout();

Быстро видно, какие признаки различают классы. Самые заметные: `Total day minutes`, `Customer service calls`, `Number vmail messages`.

> 💬 **Покажи сам:** построй отдельный boxplot для `Total day minutes` по `Churn`. Что можно сказать?

In [ ]:
# место для демонстрации

#### Scatter с цветовым кодированием категорий

In [ ]:
sns.scatterplot(
    x="Total day minutes", y="Total night minutes",
    hue="Churn", data=df, alpha=0.5
);

### 3.3 Категориальный vs. Категориальный

#### Countplot с `hue`

In [ ]:
sns.countplot(x="Customer service calls", hue="Churn", data=df);

**Наблюдение:** после 4+ звонков в поддержку резко растёт доля ушедших клиентов.

In [ ]:
_, axes = plt.subplots(1, 2, sharey=True, figsize=(12, 4))
sns.countplot(x="International plan", hue="Churn", data=df, ax=axes[0])
sns.countplot(x="Voice mail plan", hue="Churn", data=df, ax=axes[1]);

**Наблюдение:** наличие международного плана — сильный сигнал оттока. Голосовая почта — нет.

#### Таблица сопряжённости (cross tabulation)

In [ ]:
pd.crosstab(df["International plan"], df["Churn"], normalize="index").round(3)

---
## 4. Визуализация всего датасета

### 4.1 Pairplot (матрица диаграмм рассеяния)

Смотрит на все пары признаков сразу. При большом числе признаков медленный, поэтому берём подмножество.

In [ ]:
# PNG быстрее SVG для сложных графиков
%config InlineBackend.figure_format = 'png'

cols = ["Total day minutes", "Total night minutes", "Number vmail messages",
        "Customer service calls", "Churn"]

sns.pairplot(df[cols], hue="Churn", plot_kws={"alpha": 0.3});

In [ ]:
%config InlineBackend.figure_format = 'svg'

### 4.2 Heatmap

Удобен для отображения числовой переменной в разрезе двух категориальных.

In [ ]:
# Средняя продолжительность дневных звонков по State и Churn
pivot = df.pivot_table(index="Churn", values="Total day minutes", aggfunc="mean")
print(pivot)

---
## 5. Самостоятельная работа: Titanic

Данные о пассажирах Титаника. Задача — понять, какие факторы влияли на выживаемость.

Целевая переменная: `Survived` (1 — выжил, 0 — нет).

In [ ]:
train_df = pd.read_csv("../../data/titanic_train.csv", index_col="PassengerId")
train_df.head(3)

In [ ]:
train_df.describe(include="all")

In [ ]:
# Убираем Cabin (много пропусков) и строки с пропусками
train_df = train_df.drop("Cabin", axis=1).dropna()
train_df.shape

### Задание 1
Постройте pairplot для признаков `Age`, `Fare`, `SibSp`, `Parch` и `Survived`.

In [ ]:
# Ваш код здесь

### Задание 2
Как цена билета (`Fare`) зависит от класса каюты (`Pclass`)? Постройте boxplot.

In [ ]:
# Ваш код здесь

### Задание 3
Постройте тот же график, но только для значений `Fare` ниже 95-го перцентиля (чтобы убрать выбросы).

In [ ]:
# Ваш код здесь
# Подсказка: train_df["Fare"].quantile(0.95)

### Задание 4
Как зависит выживаемость от пола пассажира? Используйте `countplot` с аргументом `hue`.

In [ ]:
# Ваш код здесь

### Задание 5
Как распределялась цена билета у выживших и не выживших? Постройте `boxplot`.

In [ ]:
# Ваш код здесь

### Задание 6
Зависит ли выживаемость от возраста? Проверьте гипотезу: молодые (< 30 лет) выживали чаще, чем пожилые (> 55 лет).

*Подсказка:* создайте новый столбец `age_cat` (1 — до 30, 2 — 30–55, 3 — старше 55) и постройте `countplot` с `hue="Survived"`.

In [ ]:
# Ваш код здесь

---
## Полезные материалы

- [Документация seaborn](https://seaborn.pydata.org/api.html)
- [Галерея matplotlib](https://matplotlib.org/stable/gallery/index.html)
- Kaggle: [Titanic competition](https://www.kaggle.com/c/titanic)